In [2]:
import pandas as pd
import matplotlib.pyplot as plt 
import numpy as np
import os
import sys
import seaborn as sns
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize 
import re
import string
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from textblob import TextBlob
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from wordcloud import WordCloud
from sklearn.feature_extraction.text import TfidfVectorizer
import warnings
warnings.filterwarnings('ignore')
nltk.download('stopwords', quiet = True)
nltk.download('punkt', quiet = True)
nltk.download('punkt_tab', quiet = True)

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
sns.set_theme(style='darkgrid')

print("All libraries imported successfully!")


All libraries imported successfully!


In [3]:
print("Loading dataset...")

df_raw = pd.read_csv('../data/raw/reviews.csv')

print(f"Dataset loaded successfully!")
print(f"Rows loaded : {df_raw.shape[0]:,}")
print(f"   Columns:        {df_raw.shape[1]}")
print(f"   Memory usage:   "
      f" {df_raw.memory_usage(deep = True).sum() / 1024**2:.2f} MB")

print("\nFIRST 3 ROWS:")
print("─" * 45)
display(df_raw.head(3))

print("\nCOLUMN INFO:")
print("─" * 45)
display(df_raw.info())

Loading dataset...
Dataset loaded successfully!
Rows loaded : 568,454
   Columns:        10
   Memory usage:    424.08 MB

FIRST 3 ROWS:
─────────────────────────────────────────────


,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...



COLUMN INFO:
─────────────────────────────────────────────
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 568454 entries, 0 to 568453
Data columns (total 10 columns):
 #   Column                  Non-Null Count   Dtype 
---  ------                  --------------   ----- 
 0   Id                      568454 non-null  int64 
 1   ProductId               568454 non-null  object
 2   UserId                  568454 non-null  object
 3   ProfileName             568428 non-null  object
 4   HelpfulnessNumerator    568454 non-null  int64 
 5   HelpfulnessDenominator  568454 non-null  int64 
 6   Score                   568454 non-null  int64 
 7   Time                    568454 non-null  int64 
 8   Summary                 568427 non-null  object
 9   Text                    568454 non-null  object
dtypes: int64(5), object(5)
memory usage: 43.4+ MB


None

In [5]:
print("DATASET OVERVIEW")
print("=" * 50)

print("\n SCORE (STAR RATING) DISTRIBUTION:")
print("─" * 40)
score_counts = df_raw['Score'].value_counts().sort_index()

for star, count in score_counts.items():

    percentage = count / len(df_raw) * 100
    bar = '⭐' * star
    print(f'{bar} {star} stars:'
          f'{count:,} reviews ({percentage:.2f}%)')
    
print("\n MISSING VALUES:")
print("─" * 40)
missing = df_raw.isnull().sum()

for col, count in missing.items():
    if count > 0:
        pct = count / len(df_raw) * 100
        print(f"{col}: {count:,} missing values ({pct:.2f}%)")
    else:
        print(f"   {col}: No missing values")

print("\n REVIEW TEXT STATISTICS:")
print("─" * 40)
df_raw['text_length'] = df_raw['Text'].str.len()
df_raw['word_count'] = df_raw['Text'].str.split().str.len()

print(f"   Average review length: "
      f"{df_raw['text_length'].mean():.0f} characters")
print(f"   Average word count:    "
      f"{df_raw['word_count'].mean():.0f} words")
print(f"   Shortest review:       "
      f"{df_raw['text_length'].min()} characters")
print(f"   Longest review:        "
      f"{df_raw['text_length'].max():,} characters")

print("\n SAMPLE REVIEWS:")
print("─" * 40)

for score in [1, 3, 5]:

    sample = df_raw[df_raw['Score'] == score]['Text'].iloc[0]

    print(f"\n   ⭐ {score}-star review:")
    print(f"   '{sample[:150]}...'")

DATASET OVERVIEW

 SCORE (STAR RATING) DISTRIBUTION:
────────────────────────────────────────
⭐ 1 stars:52,268 reviews (9.19%)
⭐⭐ 2 stars:29,769 reviews (5.24%)
⭐⭐⭐ 3 stars:42,640 reviews (7.50%)
⭐⭐⭐⭐ 4 stars:80,655 reviews (14.19%)
⭐⭐⭐⭐⭐ 5 stars:363,122 reviews (63.88%)

 MISSING VALUES:
────────────────────────────────────────
   Id: No missing values
   ProductId: No missing values
   UserId: No missing values
ProfileName: 26 missing values (0.00%)
   HelpfulnessNumerator: No missing values
   HelpfulnessDenominator: No missing values
   Score: No missing values
   Time: No missing values
Summary: 27 missing values (0.00%)
   Text: No missing values
   text_length: No missing values
   word_count: No missing values

 REVIEW TEXT STATISTICS:
────────────────────────────────────────
   Average review length: 436 characters
   Average word count:    80 words
   Shortest review:       12 characters
   Longest review:        21,409 characters

 SAMPLE REVIEWS:
───────────────────────────

In [12]:
# Cell 4 — Convert Unix Timestamp to Date

print("CONVERTING TIMESTAMP TO READABLE DATE")
print("─" * 45)

print(f"Sample BEFORE conversion:")
print(f"   {df_raw['Time'].iloc[0]}")

df_raw['Date'] = pd.to_datetime(df_raw['Time'], unit='s')

df_raw['Year'] = df_raw['Date'].dt.year

df_raw['Month'] = df_raw['Date'].dt.month

df_raw['YearMonth'] = df_raw['Date'].dt.to_period('M')

print(f"\nSample AFTER conversion:")
print(f"   Date:      {df_raw['Date'].iloc[0]}")
print(f"   Year:      {df_raw['Year'].iloc[0]}")
print(f"   Month:     {df_raw['Month'].iloc[0]}")
print(f"   YearMonth: {df_raw['YearMonth'].iloc[0]}")

print(f"\nDate range of reviews:")
print(f"   Earliest: {df_raw['Date'].min().strftime('%B %Y')}")

print(f"   Latest:   {df_raw['Date'].max().strftime('%B %Y')}")
print(f"\n Date conversion complete!")

CONVERTING TIMESTAMP TO READABLE DATE
─────────────────────────────────────────────
Sample BEFORE conversion:
   1303862400

Sample AFTER conversion:
   Date:      2011-04-27 00:00:00
   Year:      2011
   Month:     4
   YearMonth: 2011-04

Date range of reviews:
   Earliest: October 1999
   Latest:   October 2012

 Date conversion complete!


In [16]:
# Cell 5 — Calculating Helpfulness Ratio

print("CALCULATING HELPFULNESS RATIO")
print("─" * 45)

df_raw['helpfulness_ratio'] = (
    df_raw['HelpfulnessNumerator'] /
    df_raw['HelpfulnessDenominator'].replace(0, 1))

print(f"Average helpfulness: "
      f"{df_raw['helpfulness_ratio'].mean():.2f}")
print(f"Reviews with votes: "
      f"{(df_raw['HelpfulnessDenominator'] > 0).sum():,}")
print(f"Reviews with no votes: "
      f"{(df_raw['HelpfulnessDenominator'] == 0).sum():,}")
print("Helpfulness ratio calculated!")


CALCULATING HELPFULNESS RATIO
─────────────────────────────────────────────
Average helpfulness: 0.41
Reviews with votes: 298,402
Reviews with no votes: 270,052
Helpfulness ratio calculated!
